# Sensorless speed estimation for a brushed DC motor

ENEL 525 course project. Mohammad Elmougey and Heinz Veintimilla.

The target is measured shaft speed. The inputs at the same instant are measured current, state-space current, and state-space speed. The state-space speed is also the baseline. Polynomials of degree 2–5 and a one-hidden-layer network are fit in NumPy. The test recording is scored only after validation chooses the model.


In [ ]:
from pathlib import Path
import json


ROOT = Path.cwd()
if not (ROOT / "results").exists():
    ROOT = Path(r"C:\Users\heinz\OneDrive\Desktop\School\ENEL525_Project")

RECOMPUTE = False
if RECOMPUTE or not (ROOT / "results" / "summary.json").exists():
    from sensorless.experiment import run_experiment
    run_experiment(ROOT, ROOT / "results")

summary = json.loads((ROOT / "results" / "summary.json").read_text(encoding="utf-8"))
print("train / val / test:", summary["n_train"], summary["n_val"], summary["n_test"])
print("validation winner:", summary["best_overall"])


## Data

Measured samples are taken at 50 µs. Model samples are taken at 1 ms. Each model time matches a measured sample, so the tables below use 40,001 aligned rows per run. Training is `t < 32` s. The scaler is fit on that slice only. Speed is reported in rad/s and current in amperes, which is the scale assumed for these traces.


In [ ]:
from IPython.display import Image, display

display(Image(filename=str(ROOT / "results" / "figures" / "eda_development.png")))
print("training means", [round(v, 3) for v in summary["scaler_mean"]])
print("training stds ", [round(v, 3) for v in summary["scaler_std"]])


## Validation

Every finite configuration is in `results/validation_results.csv`. Divergent runs are omitted from the printed table. The winner is the lowest validation RMSE.


In [ ]:
import csv
rows = list(csv.DictReader((ROOT / "results" / "validation_results.csv").open(encoding="utf-8")))
finite = [row for row in rows if row["diverged"] == "False"]
finite.sort(key=lambda row: float(row["val_rmse"]))
print(f"{'name':28} {'val_rmse':>10} {'train_rmse':>10}")
for row in finite:
    print(f"{row['name']:28} {float(row['val_rmse']):10.4f} {float(row['train_rmse']):10.4f}")
display(Image(filename=str(ROOT / "results" / "figures" / "learning_curves.png")))


## Locked test

Least squares is one solve. The selected SGD setting and the selected network are retrained with seeds 525 through 529.


In [ ]:
test = summary["test"]
print("baseline RMSE", round(test["baseline"]["rmse"], 3))
print("OLS degree", test["ols"]["degree"], "RMSE", round(test["ols"]["rmse"], 3))
print(
    "SGD degree", test["sgd"]["degree"],
    "lr", test["sgd"]["lr"],
    "RMSE", round(test["sgd"]["rmse_mean"], 3),
    "±", round(test["sgd"]["rmse_std"], 3),
)
print(
    "network width", test["mlp"]["hidden"],
    "lr", test["mlp"]["lr"],
    "RMSE", round(test["mlp"]["rmse_mean"], 3),
    "±", round(test["mlp"]["rmse_std"], 3),
)
display(Image(filename=str(ROOT / "results" / "figures" / "test_trace.png")))
display(Image(filename=str(ROOT / "results" / "figures" / "test_residual.png")))


## Reading the result

The degree-3 SGD model is the only learned model that beats the state-space speed on the locked test run. Least squares looks better on the training slice as the degree grows, and its validation error blows up; the design matrix is badly conditioned. The network reaches a lower training error and a worse test error. Its validation curve is already rising well before epoch 100, and the test recording is a smaller motion than the development recording.

The gain is also uneven in time. On the seed used for the figure, the cubic model is worse than the baseline over the first 10 seconds and better on the last 20. A speed loop would feel that plateau bias. These files are simulated, and the ampere and rad/s labels are a scale assumption.
